<a href="https://colab.research.google.com/github/bipasnadulal/aws-future-ai-programmer-nanodegree/blob/main/implementing_transfomer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [23]:
from pathlib import Path


text = Path("/content/tiny-shakespeare.txt").read_text()

In [24]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.token_id_for_char = {
            char: token_id for token_id, char in enumerate(vocabulary)
        }
        self.char_for_token_id = {
            token_id: char for token_id, char in enumerate(vocabulary)
        }

    @staticmethod
    def train_from_text(text):
        vocabulary = set(text)
        return CharTokenizer(sorted(list(vocabulary)))

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return "".join(chars)

    def vocabulary_size(self):
        return len(self.token_id_for_char)

In [25]:
tokenizer = CharTokenizer.train_from_text(text)

In [26]:
from torch.utils.data import Dataset

class TokenIdsDataset(Dataset):
  def __init__(self, data, block_size):
    self.data = data
    self.block_size = block_size

  def __len__(self):
    return len(self.data) - self.block_size

  def __getitem__(self, pos):
    assert pos < len(self.data) - self.block_size

    x = self.data[pos : pos + self.block_size]
    y = self.data[pos + 1 : pos + self.block_size + 1]

    return x, y

In [40]:
config = {
    "vocabulary_size": tokenizer.vocabulary_size(),
    "context_size":256,
    "embedding_dim":768,
    "heads_num":12,
    "layers_num":10,
    "dropout_rate":0.1,
    "use_bias":False
}

config['head_size'] = config['embedding_dim'] // config['heads_num']

In [28]:
class AttentionHead(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.Q_weights = nn.Linear(
        config['embedding_dim'], config['head_size'], config['use_bias']
    )
    self.K_weights = nn.Linear(
        config['embedding_dim'], config['head-size'], config['use_bias']
    )
    self.V_weights = nn.Linear(
        config['embedding_dim'], config['head-size'], config['use_bias']
    )
    self.dropout = nn.Dropout(config['dropout_rate'])

    casual_attention_mask = torch.tril(
        torch.ones(config['context_size'], config['context_size'])
    )
    self.register_buffer('casual_attention_mask', casual_attention_mask)

  def forward(self, input):
    batch_size, tokens_num, embedding_dim = input.shape
    Q = self.Q_weights(input)
    K = self.K_weights(input)
    V = self.V_weights(input)

    attention_scores = Q @ K.transpose(1, 2)
    attention_scores = attention_scores.masked_fill(
        self.casual_attention_mask[:tokens_num, :tokens_num] == 0,
        -torch.inf
    )
    attention_scores = attention_scores / (K.shape[-1] ** 0.5)
    attention_scores = torch.softmax(attention_scores, dim=-1)
    attention_scores = self.dropout(attention_scores)

    return attention_scores @ V

In [29]:
class AttentionHead(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.Q_weights = nn.Linear(
        config['embedding_dim'], config['head_size'], config['use_bias']
    )
    self.K_weights = nn.Linear(
        config['embedding_dim'], config['head_size'], config['use_bias']
    )
    self.V_weights = nn.Linear(
        config['embedding_dim'], config['head_size'], config['use_bias']
    )
    self.dropout = nn.Dropout(config['dropout_rate'])

    casual_attention_mask = torch.tril(
        torch.ones(config['context_size'], config['context_size'])
    )
    self.register_buffer('casual_attention_mask', casual_attention_mask)

  def forward(self, input):
    batch_size, tokens_num, embedding_dim = input.shape
    Q = self.Q_weights(input)
    K = self.K_weights(input)
    V = self.V_weights(input)

    attention_scores = Q @ K.transpose(1, 2)
    attention_scores = attention_scores.masked_fill(
        self.casual_attention_mask[:tokens_num, :tokens_num] == 0,
        -torch.inf
    )
    attention_scores = attention_scores / (K.shape[-1] ** 0.5)
    attention_scores = torch.softmax(attention_scores, dim=-1)
    attention_scores = self.dropout(attention_scores)

    return attention_scores @ V

input = torch.rand(8, config['context_size'], config['embedding_dim'])
ah = AttentionHead(config)

output = ah(input)
print(output.shape)

torch.Size([8, 256, 64])


In [30]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()

    heads_list = [AttentionHead(config) for _ in range(config['heads_num'])]
    self.heads = nn.ModuleList(heads_list)
    self.linear = nn.Linear(config['embedding_dim'], config['embedding_dim'])
    self.dropout = nn.Dropout(config['dropout_rate'])

  def forward(self, input):
    heads_outputs = [head(input) for head in self.heads]
    scores_change = torch.cat(heads_outputs, dim=-1)

    scores_change = self.linear(scores_change)

    return self.dropout(scores_change)


In [31]:
mha = MultiHeadAttention(config)
mha_output = mha(input)
print(mha_output.shape)


torch.Size([8, 256, 768])


In [32]:
class FeedForward(nn.Module):
  def __init__(self, config):
    super().__init__()

    self.linear_layers = nn.Sequential(
        nn.Linear(config['embedding_dim'], config['embedding_dim'] * 4),
        nn.GELU(),
        nn.Linear(config['embedding_dim'] * 4, config['embedding_dim']),
        nn.Dropout(config['dropout_rate'])
    )

  def forward(self, input):
    return self.linear_layers(input)


In [33]:
ff = FeedForward(config)

In [34]:
input = torch.rand(8, config["context_size"], config['embedding_dim'])

output = ff(input)

In [35]:
class Block(nn.Module):
  def __init__(self, config):
    super().__init__()

    self.multi_head = MultiHeadAttention(config)
    self.layer_norm_1 = nn.LayerNorm(config['embedding_dim'])

    self.feed_forward = FeedForward(config)
    self.layer_norm_2 = nn.LayerNorm(config['embedding_dim'])

  def forward(self, input):
    residual = input
    x = self.multi_head(self.layer_norm_1(input))
    x = x + residual

    residual = x
    x = self.feed_forward(self.layer_norm_2(x))
    x = x + residual

    return x

In [36]:
b = Block(config)

In [37]:
input = torch.rand(8, config['context_size'], config['embedding_dim'])

output = b(input)

output.shape

torch.Size([8, 256, 768])

In [38]:
from typing import Sequence
class DemoGPT(nn.Module):
  def __init__(self, config):
    super().__init__()

    self.token_embedding_layer = nn.Embedding(
        config['vocabulary_size'], config['embedding_dim']
    )
    self.positional_embedding_layer = nn.Embedding(
        config['context_size'], config['embedding']
    )

    blocks = [Block(config) for _ in range(config['layers_num'])]
    self.layers = nn.Sequential(*blocks)

    self.layer_norm = nn.LayerNorm(config['embedding_dim'])
    self.unembedding = nn.Linear(
        config['embedding_dim'], config['vocabulary_size'], bias = False
    )

  def forward(self, token_ids):
    batch_size, tokens_num = token_ids.shape

    x = self.token_embedding_layer(token_ids)
    sequence = torch.arrange(tokens_num, device = device)

    x = self.layers(x)
    x = self.layer_norm(x)
    x = self.unembedding(x)

    return x

In [41]:
from typing import Sequence

class DemoGPT(nn.Module):
  def __init__(self, config):
    super().__init__()

    self.token_embedding_layer = nn.Embedding(
        config['vocabulary_size'], config['embedding_dim']
    )
    self.positional_embedding_layer = nn.Embedding(
        config['context_size'], config['embedding_dim']
    )

    blocks = [Block(config) for _ in range(config['layers_num'])]
    self.layers = nn.Sequential(*blocks)

    self.layer_norm = nn.LayerNorm(config['embedding_dim'])
    self.unembedding = nn.Linear(
        config['embedding_dim'], config['vocabulary_size'], bias = False
    )

  def forward(self, token_ids):
    batch_size, tokens_num = token_ids.shape

    x = self.token_embedding_layer(token_ids)
    sequence = torch.arrange(tokens_num, device = device)

    x = self.layers(x)
    x = self.layer_norm(x)
    x = self.unembedding(x)

    return x

model = DemoGPT(config).to(device)

In [44]:
class DemoGPT(nn.Module):
  def __init__(self, config):
    super().__init__()

    self.token_embedding_layer = nn.Embedding(
        config['vocabulary_size'], config['embedding_dim']
    )
    self.positional_embedding_layer = nn.Embedding(
        config['context_size'], config['embedding_dim']
    )

    blocks = [Block(config) for _ in range(config['layers_num'])]
    self.layers = nn.Sequential(*blocks)

    self.layer_norm = nn.LayerNorm(config['embedding_dim'])
    self.unembedding = nn.Linear(
        config['embedding_dim'], config['vocabulary_size'], bias = False
    )

  def forward(self, token_ids):
    batch_size, tokens_num = token_ids.shape

    x = self.token_embedding_layer(token_ids)
    sequence = torch.arange(tokens_num, device = device)

    x = self.layers(x)
    x = self.layer_norm(x)
    x = self.unembedding(x)

    return x

model = DemoGPT(config).to(device)
output = model(tokenizer.encode("Hi").unsqueeze(dim=0).to(device))

In [45]:
output.shape

torch.Size([1, 2, 65])

In [46]:
def generate(model, prompt_ids, max_tokens):
    output_ids = prompt_ids
    for _ in range(max_tokens):
        if output_ids.shape[1] >= config["context_size"]:
            break
        with torch.no_grad():
            logits = model(output_ids)

        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        # Sample a random token given the softmax distribution
        next_token_id = torch.multinomial(probs, num_samples=1)
        # Add new token to the output, and repeat the process
        output_ids = torch.cat([output_ids, next_token_id], dim=-1)
    return output_ids

In [47]:
def generate_with_prompt(model, tokenizer, prompt, max_tokens=100):
    model.eval()

    prompt = tokenizer.encode(prompt).unsqueeze(dim=0).to(device)

    return tokenizer.decode(generate(model, prompt, max_tokens=max_tokens)[0])

In [48]:
generate_with_prompt(model, tokenizer, "First Citizen:\n")

"First Citizen:\n$,q:.MBF,hq'Bg;IwjAXx-VkLt$zD;IGyjp'3UNGiZqhzE;&iHrB;XkeMWefPI,,mT:km.' QQ.uE?PkU,tidr$;-hsxBgTywe 3"